## 신용카드서비스를 떠나는 고객을 찾아라
- 나이, 급여, 결혼 상태, 신용 카드 한도, 신용 카드 카테고리 등의 컬럼이 있음
- 평가: ROC-AUC, 정확도(Accuracy), F1, 정밀도(Precision), 재현율(Recall)을 구하시오
- target : Attrition_Flag (1:이탈, 0:유지)

[제출 형식]

① CSV 파일명: `result.csv`

② 이탈 확률 컬럼명: pred

③ 제출 컬럼 개수: pred 컬럼 1개

④ 평가용 데이터 개수와 예측 결과 데이터 개수 일치: 2,026개

- 제출 CSV 파일 예시
~~~
pred
0.633
0.123
0.355
~~~


### 평가(분류)
~~~
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 정확도
print(accuracy_score(y_test, pred))

# 정밀도
print(precision_score(y_test, pred))

# 재현율 (민감도)
print(recall_score(y_test, pred))

# F1
print(f1_score(y_test , pred))

# roc-auc
print(roc_auc_score(y_test, pred_proba))
~~~

In [37]:
# 1. 문제 정의
# 이탈자 예측
# y: Attrition_Flag(1: 이탈, 0: 유지)
# 이진분류모델

# 2. 데이터 불러오기
import pandas as pd
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/ex1/train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/ex1/test.csv")


In [38]:
# 3. 탐색(EDA)
print(train.shape, test.shape)

(8101, 20) (2026, 19)


In [39]:
train.describe()

,Customer_Age,Dependent_count,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Attrition_Flag
count,8101.000000,8101.000000,8101.000000,8101.000000,8101.000000,8101.000000,8101.000000,8101.000000,8101.000000,8101.000000,8101.000000,8101.000000,8101.000000,8101.000000,8101.000000
mean,46.309962,2.336502,35.904703,3.817060,2.343044,2.456240,8589.421504,1166.204543,7423.216961,0.760403,4404.021849,64.776200,0.713349,0.276823,0.158746
std,8.042270,1.308693,8.019009,1.549786,1.006502,1.106308,9049.717566,815.233038,9051.731502,0.220169,3406.953449,23.519383,0.237977,0.276527,0.365461
min,26.000000,0.000000,13.000000,1.000000,0.000000,0.000000,1438.300000,0.000000,3.000000,0.000000,530.000000,10.000000,0.000000,0.000000,0.000000
25%,41.000000,1.000000,31.000000,3.000000,2.000000,2.000000,2553.000000,432.000000,1308.000000,0.631000,2144.000000,45.000000,0.583000,0.024000,0.000000
50%,46.000000,2.000000,36.000000,4.000000,2.000000,2.000000,4503.000000,1275.000000,3432.000000,0.737000,3890.000000,67.000000,0.702000,0.179000,0.000000
75%,52.000000,3.000000,40.000000,5.000000,3.000000,3.000000,10973.000000,1787.000000,9815.000000,0.860000,4747.000000,81.000000,0.819000,0.506000,0.000000
max,73.000000,5.000000,56.000000,6.000000,6.000000,6.000000,34516.000000,2517.000000,34516.000000,3.397000,18484.000000,139.000000,3.500000,0.999000,1.000000


In [40]:
train.info()    # 결측치 없음.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8101 entries, 0 to 8100
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Customer_Age              8101 non-null   int64  
 1   Gender                    8101 non-null   object 
 2   Dependent_count           8101 non-null   int64  
 3   Education_Level           8101 non-null   object 
 4   Marital_Status            8101 non-null   object 
 5   Income_Category           8101 non-null   object 
 6   Card_Category             8101 non-null   object 
 7   Months_on_book            8101 non-null   int64  
 8   Total_Relationship_Count  8101 non-null   int64  
 9   Months_Inactive_12_mon    8101 non-null   int64  
 10  Contacts_Count_12_mon     8101 non-null   int64  
 11  Credit_Limit              8101 non-null   float64
 12  Total_Revolving_Bal       8101 non-null   int64  
 13  Avg_Open_To_Buy           8101 non-null   float64
 14  Total_Am

In [41]:
train.head()

,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Attrition_Flag
0,38,M,1,High School,Married,$80K - $120K,Blue,23,4,3,2,22149.0,987,21162.0,1.633,2409,36,0.800,0.045,0
1,44,F,3,Unknown,Divorced,Less than $40K,Blue,32,3,3,2,1693.0,1457,236.0,0.919,2829,42,0.615,0.861,1
2,49,M,4,Unknown,Unknown,$80K - $120K,Blue,39,2,2,2,2886.0,925,1961.0,0.946,5481,58,0.758,0.321,1
3,43,F,5,Graduate,Single,Unknown,Blue,34,6,6,3,4607.0,2440,2167.0,0.646,4825,77,0.878,0.530,0
4,47,M,3,High School,Married,$60K - $80K,Blue,35,5,1,2,17343.0,1280,16063.0,1.314,3170,57,0.629,0.074,0


In [42]:
# 강의 참고
# 타켓 분포 확인
train['Attrition_Flag'].value_counts()

,count
Attrition_Flag,
0,6815
1,1286


In [43]:
# 4. 전처리
# X/y split
y_train = train.pop('Attrition_Flag')
X_train = train

In [44]:
# one-hot encoding

X_train = pd.get_dummies(X_train, dtype=int)
test = pd.get_dummies(test, dtype=int)

# 추후 추가. train 기준으로 test 컬럼 맞추기
X_train, test = X_train.align(test, join='left', axis=1, fill_value=0)

In [45]:
# 5. 검증 데이터 분리
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
print(X_tr.shape, X_val.shape, y_tr.shape, y_val.shape)

(6480, 37) (1621, 37) (6480,) (1621,)


In [46]:
# 6. 학습 및 평가
# 6-1. RandomForest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_tr, y_tr)
y_pred_rf = rf.predict(X_val)

# 검증 평가
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 정확도
print(accuracy_score(y_val, y_pred_rf))

# 정밀도
print(precision_score(y_val, y_pred_rf))

# 재현율 (민감도)
print(recall_score(y_val, y_pred_rf))

# F1
print(f1_score(y_val , y_pred_rf))

# roc-auc
# 영상 참고. 확률 값으로 예측해서 사용해야 됨.
y_pred_proba_rf = rf.predict_proba(X_val)
print(roc_auc_score(y_val, y_pred_proba_rf[:, 1]))

0.9487970388648982
0.9534883720930233
0.737410071942446
0.8316430020283976
0.9865047649147993


In [47]:
# 6-2. LightGBM
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(random_state=42, verbose=-1)
lgbm.fit(X_tr, y_tr)
y_pred_lgbm = lgbm.predict(X_val)

# 정확도
print(accuracy_score(y_val, y_pred_lgbm))

# 정밀도
print(precision_score(y_val, y_pred_lgbm))

# 재현율 (민감도)
print(recall_score(y_val, y_pred_lgbm))

# F1
print(f1_score(y_val , y_pred_lgbm))

# roc-auc
y_pred_proba_lgbm = lgbm.predict_proba(X_val)
print(roc_auc_score(y_val, y_pred_proba_lgbm[:, 1]))

0.9691548426896978
0.9418604651162791
0.8741007194244604
0.9067164179104478
0.9915576102037209


In [48]:
# 7. 결과 파일
# 답지 참고
# 검증 결과 LightGBM의 성능이 더 좋으므로 최종 모델은 LightGBM.
pred = lgbm.predict_proba(test)

submit = pd.DataFrame({'pred':pred[:,1]})
submit.to_csv("result.csv", index=False)

In [49]:
# 생성한 csv 파일 다시 읽어서 확인
pd.read_csv("result.csv")

,pred
0,0.000059
1,0.001753
2,0.011551
3,0.000315
4,0.000123
...,...
2021,0.006134
2022,0.927632
2023,0.132938
2024,0.000726


### 풀이

In [50]:
# 문제정의
# 평가: ROC-AUC, 정확도(Accuracy), F1, 정밀도(Precision), 재현율(Recall)
# target: Attrition_Flag (1:이탈, 0:유지)
# 최종파일: result.csv(컬럼 1개 pred, 이탈 확률)

# 라이브러리 및 데이터 불러오기
import pandas as pd
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/ex1/train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/ex1/test.csv")

# 탐색적 데이터 분석(EDA)
print("===== 데이터 크기 =====")
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

print("\n ===== 데이터 샘플 =====")
print(train.head())
print(test.head())

print("\n ===== 데이터 정보(자료형) =====")
print(train.info())

print("\n ===== train 결측치 수 =====")
print(train.isnull().sum())
print("\n ===== test 결측치 수 =====")
print(test.isnull().sum())

print("\n ===== 타겟 분포 =====")
print(train['Attrition_Flag'].value_counts())


===== 데이터 크기 =====
Train Shape: (8101, 20)
Test Shape: (2026, 19)

 ===== 데이터 샘플 =====
   Customer_Age Gender  Dependent_count Education_Level Marital_Status  \
0            38      M                1     High School        Married   
1            44      F                3         Unknown       Divorced   
2            49      M                4         Unknown        Unknown   
3            43      F                5        Graduate         Single   
4            47      M                3     High School        Married   

  Income_Category Card_Category  Months_on_book  Total_Relationship_Count  \
0    $80K - $120K          Blue              23                         4   
1  Less than $40K          Blue              32                         3   
2    $80K - $120K          Blue              39                         2   
3         Unknown          Blue              34                         6   
4     $60K - $80K          Blue              35                         5   

   Mo

In [51]:
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/ex1/train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/refs/heads/main/p2/ex1/test.csv")

# 데이터 전처리
target = train.pop('Attrition_Flag')
# print(train.shape, test.shape)

# 인코딩
cols = train.select_dtypes(include='object').columns
print(cols)


# label인코딩
# from sklearn.preprocessing import LabelEncoder

# for col in cols:
#     le = LabelEncoder()
#     train[col] = le.fit_transform(train[col])
#     test[col] = le.transform(test[col])



# one-hot
print(train.shape, test.shape)
train = pd.get_dummies(train)
test = pd.get_dummies(test)
print(train.shape, test.shape)


Index(['Gender', 'Education_Level', 'Marital_Status', 'Income_Category',
       'Card_Category'],
      dtype='object')
(8101, 19) (2026, 19)
(8101, 37) (2026, 37)


In [52]:
# 검증데이터 분리
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=2022)
print(X_tr.shape, X_val.shape, y_tr.shape, y_val.shape)

# 랜덤포레스트
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(random_state=2022)
model.fit(X_tr, y_tr)
pred = model.predict(X_val)

# 검증 평가
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
print("랜덤포레스트")

# 정확도
print(accuracy_score(y_val, pred))

# 정밀도
print(precision_score(y_val, pred))

# 재현율 (민감도)
print(recall_score(y_val, pred))

# F1
print(f1_score(y_val , pred))

# roc-auc
pred = model.predict_proba(X_val)
print(roc_auc_score(y_val, pred[:,1]))


(6480, 37) (1621, 37) (6480,) (1621,)
랜덤포레스트
0.9605181986428131
0.9419642857142857
0.8053435114503816
0.8683127572016461
0.9885945548197205


In [53]:
# LightGBM
import lightgbm as lgb
lg = lgb.LGBMClassifier(random_state=2022, verbose=-1)
lg.fit(X_tr, y_tr)
pred = lg.predict(X_val)

print("lightgbm")
# 정확도
print(accuracy_score(y_val, pred))

# 정밀도
print(precision_score(y_val, pred))

# 재현율 (민감도)
print(recall_score(y_val, pred))

# F1
print(f1_score(y_val , pred))

# roc-auc
pred = lg.predict_proba(X_val)
print(roc_auc_score(y_val, pred[:,1]))


lightgbm
0.9747069710055521
0.9333333333333333
0.9083969465648855
0.9206963249516441
0.9940318712119935


In [54]:
# 예측 및 csv 제출
pred = lg.predict_proba(test)

result = pd.DataFrame({'pred': pred[:,1]})
result.to_csv("result.csv", index=False)


In [55]:
# 1. pred 행의 수와 test의 행의 수 비교
print("pred: ", pred.shape, " test: ", test.shape)

# 2. 생성한 csv 를 다시 읽어 확인
result = pd.read_csv("result.csv")
print(result.head())


pred:  (2026, 2)  test:  (2026, 37)
       pred
0  0.000069
1  0.005550
2  0.012855
3  0.000170
4  0.000230


In [56]:
# 심화 - 합쳐서 인코딩

# 합쳐서 인코딩
n = len(train)
df = pd.concat([train, test])
cols = df.select_dtypes(include='object').columns

# label
from sklearn.preprocessing import LabelEncoder
for col in cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

# # one-hot
# df = pd.get_dummies(df, columns=cols)

# 다시 나누기
train = df.iloc[:n]
test = df.iloc[n:]
print(train.shape, test.shape)

(8101, 37) (2026, 37)
